In [1]:
import sys
# TO CHANGE
BASEDIR = "../../../.."
sys.path.insert(0, BASEDIR)

In [2]:
from pprint import pprint

In [3]:
from src.db_drivers.vector_driver import EmbedderModelConfig
from src.kg_model import KnowledgeGraphModel, KnowledgeGraphModelConfig
from src.agents import AgentDriverConfig
from src.utils import TripletCreator, NodeCreator, RelationCreator
from src.agents.configs import DEFAULT_AGENT_CONFIGS
from src.kg_model.utils import KGEmbeddersMapping, AgentsMapping

/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Пример работы с моделью графа знаний

1. Инициализация класса, выполняющего функции embedder-модели

In [4]:
EMBEDDER_MODEL_PATH = '../../../../models/intfloat/multilingual-e5-base' # PATH TO APPROPRIATE EMBEDDER-MODEL
embedder_config = EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH)
pprint(embedder_config)

EmbedderModelConfig(model_name_or_path='../../../../models/intfloat/multilingual-e5-base',
                    prompts={'passage': 'passage: ', 'query': 'query: '},
                    query_prompt_name='query',
                    passage_prompt_name='passage',
                    device='cuda',
                    normalize_embeddings=True)


2. Инициализация Connector-класса к LLM-моделе

In [5]:
adriver_config = AgentDriverConfig(name='ollama', agent_config=DEFAULT_AGENT_CONFIGS['ollama'])
pprint(adriver_config, width=200)

AgentDriverConfig(name='ollama',
                  agent_config=AgentConnectorConfig(gen_strategy={'num_predict': 2048, 'seed': 42, 'temperature': 0.0, 'top_k': 1},
                                                    credentials={'host': 'localhost', 'model': 'llama3.1:8b', 'port': 11437},
                                                    ext_params={'keep_alive': -1, 'timeout': 560, 'trials': 5}))


3. Спецификация embedder- и llm-моделей, которые будут использоваться в компонентах модели графа знаний

In [6]:
EMBEDDERS_CONFIG = {
    'm-e5-base': embedder_config
}

EMBEDDERS_MAP = KGEmbeddersMapping(
    embeddings_model={'dense_nodes': 'm-e5-base', 'dense_triplets': 'm-e5-base'}
)

AGENTS_CONFIG = {
    'llama3.1:8b': adriver_config 
}

AGENTS_MAP = AgentsMapping(
    qa_pipeline='llama3.1:8b',
    mem_pipeline='llama3.1:8b',
    kg_nodestree_model='llama3.1:8b'
)

4. Инициализация модели графа знаний

In [7]:
kg_config = KnowledgeGraphModelConfig(
    agents_configs=AGENTS_CONFIG,
    embedders_configs=EMBEDDERS_CONFIG,
    agents_map=AGENTS_MAP,
    embedders_map=EMBEDDERS_MAP,
    nodestree_config=None # Модель дерева вершин строиться не будет
) 

In [8]:
kg_model = KnowledgeGraphModel(kg_config)

No sentence-transformers model found with name ../../../../models/intfloat/multilingual-e5-base. Creating a new one with mean pooling.


In [9]:
pprint(kg_model.count_items(detailed=True))

{'embeddings_info': {'nodes': {'episodic': {'bm25_nodes': 0, 'dense_nodes': 0},
                               'hyper': {'bm25_nodes': 0, 'dense_nodes': 0},
                               'object': {'bm25_nodes': 0, 'dense_nodes': 4},
                               'time': {'bm25_nodes': 0, 'dense_nodes': 0}},
                     'triplets': {'bm25_triplets': 0, 'dense_triplets': 2}},
 'graph_info': {'nodes': {'episodic': 0, 'hyper': 0, 'object': 4, 'time': 0},
                'triplets': {'episodic': 0,
                             'hyper': 0,
                             'simple': 2,
                             'time': 0}},
 'nodestree_info': None}


5. Добавление триплетов в модель графа знаний

In [10]:
TRIPLET_EXAMPLE1 = TripletCreator.create(
    start_node=NodeCreator.create(n_type='object', name='саша'), 
    relation=RelationCreator.create(r_type='simple', name='шла'), 
    end_node=NodeCreator.create(n_type='object', name='улица'))
TRIPLET_EXAMPLE2 = TripletCreator.create(
    start_node=NodeCreator.create(n_type='object', name='маша'), 
    relation=RelationCreator.create(r_type='simple', name='шла'), 
    end_node=NodeCreator.create(n_type='object', name='шоссе'))
TRIPLET_EXAMPLE3 = TripletCreator.create(
    start_node=NodeCreator.create(n_type='object', name='теплоход'), 
    relation=RelationCreator.create(r_type='simple', name='плыл'), 
    end_node=NodeCreator.create(n_type='object', name='канал'))
TRIPLET_EXAMPLE4 = TripletCreator.create(
    start_node=NodeCreator.create(n_type='object', name='моторная лодка'), 
    relation=RelationCreator.create(r_type='simple', name='плыл'), 
    end_node=NodeCreator.create(n_type='object', name='река'))

In [11]:
add_info, trace = kg_model.add_knowledge([TRIPLET_EXAMPLE1, TRIPLET_EXAMPLE2, TRIPLET_EXAMPLE3, TRIPLET_EXAMPLE4], check_consistency=True, status_bar=True)
print("Информация о добавленных триплетах: ")
pprint(add_info)
print("Количество элементов в моделе графа знаний:")
pprint(kg_model.count_items())

100%|██████████| 1/1 [00:02<00:00,  2.77s/it]

Информация о добавленных триплетах: 
{'embeddings_info': {'nodes': {<NodeType.time: 'time'>: set(),
                               <NodeType.episodic: 'episodic'>: set(),
                               <NodeType.hyper: 'hyper'>: set(),
                               <NodeType.object: 'object'>: {'2401d7e9c775689eceb9108c20060b0e',
                                                             '64ec88e2cb968f3b8c55f5e7316f0418',
                                                             '72adc4fc2f53b1344fb5b56d6ed81b1f',
                                                             '956c2d5e231f655cee06efa6a32969e9',
                                                             'b8f8c47ba32fcbf247f40e993965ad01',
                                                             'd51ca416731c328c6bb159cc30803659',
                                                             'eebef31909a5d48ab3d1b7f3cd507df4',
                                                             'f7670dc5605796e6f98a48a

In [12]:
pprint(trace)

SimpleModuleResult(context={'check_consistency': True,
                            'positional_arguments': ([Triplet(start_node=Node(name='саша',
                                                                              type=<NodeType.object: 'object'>,
                                                                              prop={},
                                                                              stringified='саша',
                                                                              id='72adc4fc2f53b1344fb5b56d6ed81b1f'),
                                                              relation=Relation(name='шла',
                                                                                type=<RelationType.simple: 'simple'>,
                                                                                prop={},
                                                                                id='c2c994676e30850fdf95f0f2a647ba30'),
                    

6. Удаление триплетов из модели графа знаний

In [13]:
remove_info, trace = kg_model.remove_knowledge([TRIPLET_EXAMPLE2, TRIPLET_EXAMPLE4])
print("Информация об удалённых триплетах: ")
pprint(remove_info)
print("Количество элементов в моделе графа знаний:")
pprint(kg_model.count_items())

Информация об удалённых триплетах: 
{'embeddings_info': {0: {'e_node': True, 's_node': True, 'triplet': True},
                     1: {'e_node': True, 's_node': True, 'triplet': True}},
 'graph_info': {0: {'e_node': True, 's_node': True},
                1: {'e_node': True, 's_node': True}},
 'tree_info': None}
Количество элементов в моделе графа знаний:
{'embeddings_info': {'nodes': 4, 'triplets': 2},
 'graph_info': {'nodes': 4, 'triplets': 2},
 'nodestree_info': None}


In [14]:
pprint(trace)

SimpleModuleResult(context={'positional_arguments': ([Triplet(start_node=Node(name='маша',
                                                                              type=<NodeType.object: 'object'>,
                                                                              prop={},
                                                                              stringified='маша',
                                                                              id='d51ca416731c328c6bb159cc30803659'),
                                                              relation=Relation(name='шла',
                                                                                type=<RelationType.simple: 'simple'>,
                                                                                prop={},
                                                                                id='8c3274995b6a321367f6986fe3f1bace'),
                                                              end_node=Node

7. Полное удаление содержимого модели графа знаний

In [15]:
kg_model.clear()
print("Количество элементов в моделе графа знаний:")
pprint(kg_model.count_items())
print("Консистентность модели графа знаний:", kg_model.check_consistency())

Количество элементов в моделе графа знаний:
{'embeddings_info': {'nodes': 0, 'triplets': 0},
 'graph_info': {'nodes': 0, 'triplets': 0},
 'nodestree_info': None}
Консистентность модели графа знаний: True


In [ ]:
kg_model.close_connections()
del kg_model

: 